# Phase 3 — Autoencoder: Unsupervised Anomaly Detection
**Goal:** Train an autoencoder exclusively on BENIGN traffic so that it learns a compact representation of normal behaviour. At inference, flows that produce high reconstruction error are flagged as anomalies — including attack types never seen during training.

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, roc_curve,
                             accuracy_score, precision_score, recall_score, f1_score)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
import joblib, os, warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

DATA_PATH   = r"C:\Users\hp\Desktop\Projets\cyber-ids-project\data"
MODELS_PATH = r"C:\Users\hp\Desktop\Projets\cyber-ids-project\models"
PLOTS_PATH  = r"C:\Users\hp\Desktop\Projets\cyber-ids-project\reports"
os.makedirs(MODELS_PATH, exist_ok=True)

## 2. Load Data

In [ ]:
df = pd.read_parquet(os.path.join(DATA_PATH, 'clean_data.parquet'))
le = joblib.load(os.path.join(DATA_PATH, 'label_encoder.pkl'))

X = df.drop('Label', axis=1).values
y = df['Label'].values
n_features = X.shape[1]
print(f"✅ {len(X):,} rows  ·  {n_features} features")

## 3. Prepare Autoencoder Dataset

In [ ]:
benign_idx = np.where(le.classes_ == 'BENIGN')[0][0]

X_benign = X[y == benign_idx];  y_benign = np.zeros(len(X_benign))
X_attack = X[y != benign_idx];  y_attack = np.ones(len(X_attack))

print(f"BENIGN : {len(X_benign):,}  |  ATTACK : {len(X_attack):,}")

X_train_ae, X_test_benign = train_test_split(X_benign, test_size=0.2, random_state=42)

X_test_ae = np.vstack([X_test_benign, X_attack])
y_test_ae = np.hstack([y_benign[:len(X_test_benign)], y_attack])

print(f"AE training set : {X_train_ae.shape[0]:,} (BENIGN only)")
print(f"AE test set     : {X_test_ae.shape[0]:,} (BENIGN + ATTACK)")

## 4. Autoencoder Architecture

**Encoder:** 66 → 64 → 32 → **16 (bottleneck)**  
**Decoder:** 16 → 32 → 64 → 66  
Loss function: Mean Squared Error (reconstruction quality)

In [ ]:
def build_autoencoder(n_features):
    encoder = keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(), layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(16, activation='relu'),
    ], name='encoder')

    decoder = keras.Sequential([
        layers.Input(shape=(16,)),
        layers.Dense(32, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(), layers.Dropout(0.2),
        layers.Dense(n_features, activation='linear'),
    ], name='decoder')

    inp     = keras.Input(shape=(n_features,))
    encoded = encoder(inp)
    decoded = decoder(encoded)
    ae = keras.Model(inp, decoded, name='autoencoder')
    ae.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
    return ae, encoder

autoencoder, encoder = build_autoencoder(n_features)
autoencoder.summary()

## 5. Training

In [ ]:
cb_ae = [
    callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
    callbacks.ModelCheckpoint(os.path.join(MODELS_PATH, 'autoencoder_checkpoint.keras'),
                              monitor='val_loss', save_best_only=True, verbose=1)
]

ae_history = autoencoder.fit(
    X_train_ae, X_train_ae,
    epochs=50, batch_size=2048, validation_split=0.1,
    shuffle=True, callbacks=cb_ae, verbose=1)

plt.figure(figsize=(10, 5))
plt.plot(ae_history.history['loss'],     label='Train', color='#3498db')
plt.plot(ae_history.history['val_loss'], label='Val',   color='#e74c3c')
plt.title('Autoencoder Training Loss', fontsize=14, fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(PLOTS_PATH, 'autoencoder_training_loss.png'), dpi=150)
plt.show()

## 6. Anomaly Threshold

In [ ]:
X_train_sample   = X_train_ae[:50000]
train_recon      = autoencoder.predict(X_train_sample, verbose=0)
train_errors     = np.mean(np.power(X_train_sample - train_recon, 2), axis=1)
threshold        = np.percentile(train_errors, 95)
print(f"Anomaly threshold (95th percentile of BENIGN error): {threshold:.6f}")

## 7. Evaluation

In [ ]:
test_recon  = autoencoder.predict(X_test_ae, verbose=0)
test_errors = np.mean(np.power(X_test_ae - test_recon, 2), axis=1)
y_pred_ae   = (test_errors > threshold).astype(int)

acc  = accuracy_score(y_test_ae, y_pred_ae)
prec = precision_score(y_test_ae, y_pred_ae, zero_division=0)
rec  = recall_score(y_test_ae, y_pred_ae, zero_division=0)
f1   = f1_score(y_test_ae, y_pred_ae, zero_division=0)
print(f"Accuracy: {acc:.4f}  |  Precision: {prec:.4f}  |  Recall: {rec:.4f}  |  F1: {f1:.4f}")
print(classification_report(y_test_ae, y_pred_ae, target_names=['Normal', 'Anomaly'], zero_division=0))

### ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test_ae, test_errors)
auc_score   = roc_auc_score(y_test_ae, test_errors)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#e74c3c', lw=2, label=f'AUC = {auc_score:.4f}')
plt.plot([0,1],[0,1], color='gray', linestyle='--', label='Random')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('Autoencoder ROC Curve', fontsize=14, fontweight='bold')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(PLOTS_PATH, 'autoencoder_roc.png'), dpi=150)
plt.show()
print(f"AUC: {auc_score:.4f}")

### Reconstruction Error Distribution

In [ ]:
benign_errors = test_errors[:len(X_test_benign)]
attack_errors = test_errors[len(X_test_benign):]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, log, title in [(axes[0], True, 'Log Scale'), (axes[1], False, 'Zoomed')]:
    ax.hist(benign_errors, bins=100, alpha=0.7, color='#2ecc71',
            label=f'Normal (mean={np.mean(benign_errors):.3f})', density=True)
    ax.hist(attack_errors, bins=100, alpha=0.7, color='#e74c3c',
            label=f'Attack (mean={np.mean(attack_errors):.3f})', density=True)
    ax.axvline(threshold, color='black', linestyle='--', lw=2,
               label=f'Threshold = {threshold:.4f}')
    if log: ax.set_yscale('log')
    else:   ax.set_xlim(0, np.percentile(benign_errors, 99))
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Reconstruction Error (MSE)'); ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Autoencoder Reconstruction Error Distribution', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_PATH, 'autoencoder_error_distribution.png'), dpi=150)
plt.show()

## 8. Save Model & Threshold

In [ ]:
autoencoder.save(os.path.join(MODELS_PATH, 'autoencoder.keras'))
joblib.dump(threshold, os.path.join(MODELS_PATH, 'autoencoder_threshold.pkl'))
print("✅ autoencoder.keras  saved")
print("✅ autoencoder_threshold.pkl  saved")